# Compare Models on the Same Questions

This notebook aligns two offline latent-collection runs by `uid`, so you can compare different models on exactly the same questions.

It does five things:
1. loads `metadata.jsonl` from two runs,
2. intersects them on `uid`,
3. summarizes correctness overlap,
4. compares simple latent features for matched questions,
5. prints side-by-side examples where the models differ.

Use this when you want to compare, for example:
- `Qwen/Qwen3-4B`
- `allenai/Llama-3.1-Tulu-3-8B`

on the same MATH-500 examples.

In [ ]:
from pathlib import Path
import json
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch

from research.buffer import ParquetLatentBuffer
from research.latent.codec import deserialize_latent_tensor

RUN_A = Path('/workspace/outputs/qwen3_4b_math500_latents_pilot_4096_500')
RUN_B = Path('/workspace/outputs/tulu3_math500_latents_pilot_4096_500')
LABEL_A = 'Qwen3-4B'
LABEL_B = 'Tulu3-8B'
USE_REASONING_SPAN = True
MAX_MATCHED_ROWS = None
RANDOM_SEED = 1
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

def load_jsonl(path: Path):
    rows = []
    with path.open(encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def load_metadata(run_dir: Path):
    path = run_dir / 'metadata.jsonl'
    assert path.exists(), f'missing metadata: {path}'
    rows = load_jsonl(path)
    rows_by_uid = {row['uid']: row for row in rows}
    return rows, rows_by_uid


rows_a, meta_a = load_metadata(RUN_A)
rows_b, meta_b = load_metadata(RUN_B)
common_uids = sorted(set(meta_a) & set(meta_b))
if MAX_MATCHED_ROWS is not None:
    common_uids = common_uids[:MAX_MATCHED_ROWS]

print(f'{LABEL_A}: {len(rows_a)} rows')
print(f'{LABEL_B}: {len(rows_b)} rows')
print(f'Matched on uid: {len(common_uids)} rows')

In [ ]:
def load_buffer_rows(run_dir: Path):
    buffer_dir = run_dir / 'buffer'
    buffer = ParquetLatentBuffer(
        root_dir=str(buffer_dir),
        shard_max_samples=256,
        compression='zstd',
        max_disk_gb=1000.0,
    )
    rows = list(buffer.iter_rows())
    return {row['uid']: row for row in rows}

buffer_a = load_buffer_rows(RUN_A)
buffer_b = load_buffer_rows(RUN_B)
print(f'{LABEL_A} latent rows:', len(buffer_a))
print(f'{LABEL_B} latent rows:', len(buffer_b))

In [ ]:
def extract_sequence_for_analysis(buffer_row, meta_row, use_reasoning_span=True):
    seq = deserialize_latent_tensor(
        buffer_row['latent_blob'],
        seq_len=int(buffer_row['response_length']),
        hidden_dim=int(buffer_row['hidden_dim']),
        dtype=str(buffer_row['dtype']),
    ).float()

    start = 0
    end = seq.shape[0]
    if use_reasoning_span and meta_row.get('think_token_start') is not None and meta_row.get('think_token_end') is not None:
        start = int(meta_row['think_token_start'])
        end = int(meta_row['think_token_end'])
        start = max(0, min(start, seq.shape[0]))
        end = max(start + 1, min(end, seq.shape[0]))
    return seq[start:end]


def sample_features(uid, meta_row, buffer_row):
    seq = extract_sequence_for_analysis(buffer_row, meta_row, use_reasoning_span=USE_REASONING_SPAN)
    pooled_mean = seq.mean(dim=0)
    pooled_last = seq[-1]
    step_deltas = seq[1:] - seq[:-1] if seq.shape[0] > 1 else torch.zeros_like(seq[:1])
    return {
        'uid': uid,
        'success': bool(meta_row['success']),
        'score_accuracy': float(meta_row['score_accuracy']),
        'response_length': int(meta_row['response_length']),
        'analysis_length': int(seq.shape[0]),
        'has_think_tags': bool(meta_row.get('has_think_tags', False)),
        'path_length': float(step_deltas.norm(dim=-1).sum().item()) if seq.shape[0] > 1 else 0.0,
        'mean_norm': float(seq.norm(dim=-1).mean().item()),
        'mean_feature': pooled_mean.numpy(),
        'last_feature': pooled_last.numpy(),
        'question': meta_row['question'],
        'response': meta_row['response'],
        'final_answer_text': meta_row.get('final_answer_text'),
    }

matched = []
for uid in common_uids:
    if uid not in buffer_a or uid not in buffer_b:
        continue
    matched.append({
        'uid': uid,
        'a': sample_features(uid, meta_a[uid], buffer_a[uid]),
        'b': sample_features(uid, meta_b[uid], buffer_b[uid]),
    })

print('Matched examples with latent rows:', len(matched))

In [ ]:
def outcome_bucket(item):
    a = item['a']['success']
    b = item['b']['success']
    if a and b:
        return 'both_correct'
    if (not a) and (not b):
        return 'both_wrong'
    if a and (not b):
        return f'{LABEL_A}_only_correct'
    return f'{LABEL_B}_only_correct'

bucket_counts = Counter(outcome_bucket(item) for item in matched)
print('Outcome overlap:')
for key, value in sorted(bucket_counts.items()):
    print(f'  {key}: {value}')

In [ ]:
def summarize(values):
    arr = np.asarray(values, dtype=np.float64)
    return {
        'n': int(arr.size),
        'mean': float(arr.mean()) if arr.size else None,
        'std': float(arr.std()) if arr.size else None,
    }

for metric in ['response_length', 'analysis_length', 'path_length', 'mean_norm']:
    print(f'Metric: {metric}')
    print(f'  {LABEL_A}:', summarize([item['a'][metric] for item in matched]))
    print(f'  {LABEL_B}:', summarize([item['b'][metric] for item in matched]))

In [ ]:
def pca_project(x, n_components=3):
    x = np.asarray(x, dtype=np.float32)
    x_centered = x - x.mean(axis=0, keepdims=True)
    _, s, vt = np.linalg.svd(x_centered, full_matrices=False)
    coords = x_centered @ vt[:n_components].T
    explained = (s ** 2) / max(1, x.shape[0] - 1)
    explained_ratio = explained / explained.sum()
    return coords, explained_ratio[:n_components]

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401


def plot_success_failure_scatter_3d(coords, labels, title, var_ratio=None):
    fig = plt.figure(figsize=(8, 7))
    ax = fig.add_subplot(111, projection='3d')
    failure = labels == 0
    success = labels == 1
    if failure.any():
        ax.scatter(coords[failure, 0], coords[failure, 1], coords[failure, 2], s=30, alpha=0.75, c='#d1495b', label='failure')
    if success.any():
        ax.scatter(coords[success, 0], coords[success, 1], coords[success, 2], s=30, alpha=0.75, c='#2a9d8f', label='success')
    ax.set_title(title)
    if var_ratio is None or len(var_ratio) < 3:
        ax.set_xlabel('PC1')
        ax.set_ylabel('PC2')
        ax.set_zlabel('PC3')
    else:
        ax.set_xlabel(f'PC1 ({var_ratio[0] * 100:.1f}% var)')
        ax.set_ylabel(f'PC2 ({var_ratio[1] * 100:.1f}% var)')
        ax.set_zlabel(f'PC3 ({var_ratio[2] * 100:.1f}% var)')
    ax.legend()
    plt.show()


In [ ]:
mean_a = np.stack([item['a']['mean_feature'] for item in matched], axis=0)
mean_b = np.stack([item['b']['mean_feature'] for item in matched], axis=0)
last_a = np.stack([item['a']['last_feature'] for item in matched], axis=0)
last_b = np.stack([item['b']['last_feature'] for item in matched], axis=0)

labels_a = np.asarray([int(item['a']['success']) for item in matched])
labels_b = np.asarray([int(item['b']['success']) for item in matched])

mean_coords_a, mean_var_a = pca_project(mean_a)
mean_coords_b, mean_var_b = pca_project(mean_b)
last_coords_a, last_var_a = pca_project(last_a)
last_coords_b, last_var_b = pca_project(last_b)

plot_success_failure_scatter_3d(mean_coords_a, labels_a, title=f'{LABEL_A}: mean-pooled trajectory PCA', var_ratio=mean_var_a)
plot_success_failure_scatter_3d(mean_coords_b, labels_b, title=f'{LABEL_B}: mean-pooled trajectory PCA', var_ratio=mean_var_b)
plot_success_failure_scatter_3d(last_coords_a, labels_a, title=f'{LABEL_A}: last-token trajectory PCA', var_ratio=last_var_a)
plot_success_failure_scatter_3d(last_coords_b, labels_b, title=f'{LABEL_B}: last-token trajectory PCA', var_ratio=last_var_b)


## Cross-Model Last-Token Alignment with CCA

This section uses matched-question **last-token** features from the two models, learns a shared CCA space, and then runs PCA on the aligned features.

Why this is valid:
- the raw hidden sizes differ across models,
- CCA finds paired linear directions that are maximally correlated across matched questions,
- PCA on the aligned outputs gives a shared visualization space.


In [ ]:
def _sym_inv_sqrt(mat, eps=1e-5):
    eigvals, eigvecs = np.linalg.eigh(mat)
    eigvals = np.clip(eigvals, eps, None)
    inv_sqrt = eigvecs @ np.diag(1.0 / np.sqrt(eigvals)) @ eigvecs.T
    return inv_sqrt


def cca_align(x, y, n_components=None, reg=1e-3):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    assert x.shape[0] == y.shape[0], 'CCA requires matched rows'
    n = x.shape[0]
    if n_components is None:
        n_components = min(16, x.shape[1], y.shape[1], max(1, n - 1))
    n_components = min(n_components, x.shape[1], y.shape[1], max(1, n - 1))

    x_mean = x.mean(axis=0, keepdims=True)
    y_mean = y.mean(axis=0, keepdims=True)
    xc = x - x_mean
    yc = y - y_mean

    scale = max(1, n - 1)
    sxx = (xc.T @ xc) / scale + reg * np.eye(x.shape[1])
    syy = (yc.T @ yc) / scale + reg * np.eye(y.shape[1])
    sxy = (xc.T @ yc) / scale

    sxx_inv_sqrt = _sym_inv_sqrt(sxx)
    syy_inv_sqrt = _sym_inv_sqrt(syy)
    tmat = sxx_inv_sqrt @ sxy @ syy_inv_sqrt

    u, s, vt = np.linalg.svd(tmat, full_matrices=False)
    wx = sxx_inv_sqrt @ u[:, :n_components]
    wy = syy_inv_sqrt @ vt.T[:, :n_components]

    zx = xc @ wx
    zy = yc @ wy
    return {
        'zx': zx.astype(np.float32),
        'zy': zy.astype(np.float32),
        'corrs': s[:n_components].astype(np.float32),
        'x_mean': x_mean.astype(np.float32),
        'y_mean': y_mean.astype(np.float32),
        'wx': wx.astype(np.float32),
        'wy': wy.astype(np.float32),
    }


cca_result = cca_align(last_a, last_b, n_components=min(8, len(matched) - 1 if len(matched) > 1 else 1))
print('CCA canonical correlations:', cca_result['corrs'])

aligned_last_a = cca_result['zx']
aligned_last_b = cca_result['zy']
combined_aligned_last = np.concatenate([aligned_last_a, aligned_last_b], axis=0)
aligned_coords, aligned_var = pca_project(combined_aligned_last)
na = aligned_last_a.shape[0]

fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection='3d')

success_a = labels_a == 1
failure_a = labels_a == 0
success_b = labels_b == 1
failure_b = labels_b == 0

ax.scatter(aligned_coords[:na][failure_a, 0], aligned_coords[:na][failure_a, 1], aligned_coords[:na][failure_a, 2],
           c='#1f77b4', marker='x', s=40, alpha=0.8, label=f'{LABEL_A} failure')
ax.scatter(aligned_coords[:na][success_a, 0], aligned_coords[:na][success_a, 1], aligned_coords[:na][success_a, 2],
           c='#1f77b4', marker='o', s=40, alpha=0.8, label=f'{LABEL_A} success')
ax.scatter(aligned_coords[na:][failure_b, 0], aligned_coords[na:][failure_b, 1], aligned_coords[na:][failure_b, 2],
           c='#ff7f0e', marker='x', s=40, alpha=0.8, label=f'{LABEL_B} failure')
ax.scatter(aligned_coords[na:][success_b, 0], aligned_coords[na:][success_b, 1], aligned_coords[na:][success_b, 2],
           c='#ff7f0e', marker='o', s=40, alpha=0.8, label=f'{LABEL_B} success')

ax.set_title('CCA-aligned last-token PCA')
ax.set_xlabel(f'PC1 ({aligned_var[0] * 100:.1f}% var)')
ax.set_ylabel(f'PC2 ({aligned_var[1] * 100:.1f}% var)')
ax.set_zlabel(f'PC3 ({aligned_var[2] * 100:.1f}% var)')
ax.legend(loc='best')
plt.show()


## Linear Separability with a Linear SVM

This section measures how well success vs failure can be linearly separated.

It tries to use `sklearn` if available and reports 5-fold cross-validated accuracy for:
- mean-pooled trajectory features,
- last-token features,
- CCA-aligned last-token features.

If `sklearn` is unavailable in the container, the cell will print a message and skip cleanly.


In [ ]:
try:
    from sklearn.model_selection import StratifiedKFold, cross_val_score
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.svm import LinearSVC
    have_sklearn = True
except Exception as exc:
    have_sklearn = False
    print('sklearn is unavailable:', exc)


def run_linear_svm_cv(x, y, name):
    y = np.asarray(y, dtype=np.int64)
    class_counts = np.bincount(y)
    min_class = class_counts.min() if class_counts.size > 1 else 0
    if min_class < 2:
        print(f'{name}: skipped, not enough examples per class for CV')
        return None

    n_splits = min(5, int(min_class))
    clf = make_pipeline(
        StandardScaler(),
        LinearSVC(C=1.0, dual='auto', max_iter=20000, random_state=RANDOM_SEED),
    )
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    scores = cross_val_score(clf, x, y, cv=cv, scoring='accuracy')
    print(f'{name}: accuracy mean={scores.mean():.4f}, std={scores.std():.4f}, folds={n_splits}')
    return scores


if have_sklearn:
    run_linear_svm_cv(mean_a, labels_a, f'{LABEL_A} mean feature')
    run_linear_svm_cv(last_a, labels_a, f'{LABEL_A} last token')
    run_linear_svm_cv(mean_b, labels_b, f'{LABEL_B} mean feature')
    run_linear_svm_cv(last_b, labels_b, f'{LABEL_B} last token')

    if 'aligned_last_a' in globals() and 'aligned_last_b' in globals():
        run_linear_svm_cv(aligned_last_a, labels_a, f'{LABEL_A} CCA-aligned last token')
        run_linear_svm_cv(aligned_last_b, labels_b, f'{LABEL_B} CCA-aligned last token')


In [ ]:
paired_deltas = {
    'response_length_delta': np.asarray([item['a']['response_length'] - item['b']['response_length'] for item in matched], dtype=np.float32),
    'analysis_length_delta': np.asarray([item['a']['analysis_length'] - item['b']['analysis_length'] for item in matched], dtype=np.float32),
    'path_length_delta': np.asarray([item['a']['path_length'] - item['b']['path_length'] for item in matched], dtype=np.float32),
    'mean_norm_delta': np.asarray([item['a']['mean_norm'] - item['b']['mean_norm'] for item in matched], dtype=np.float32),
}

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes = axes.ravel()
for ax, (name, values) in zip(axes, paired_deltas.items()):
    ax.hist(values, bins=30, color='#457b9d', alpha=0.8)
    ax.axvline(0.0, color='black', linestyle='--', linewidth=1)
    ax.set_title(name)
    ax.set_xlabel(f'{LABEL_A} - {LABEL_B}')
    ax.set_ylabel('count')
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

for name, values in paired_deltas.items():
    print(name, summarize(values))


## Inspect disagreement cases

This cell prints examples where one model gets the question right and the other gets it wrong.

In [ ]:
def show_disagreements(items, n=5):
    shown = 0
    for item in items:
        a_ok = item['a']['success']
        b_ok = item['b']['success']
        if a_ok == b_ok:
            continue
        print('=' * 140)
        print('uid:', item['uid'])
        print('question:', item['a']['question'])
        print(f'{LABEL_A}: success={a_ok}, answer={item["a"]["final_answer_text"]}, response_length={item["a"]["response_length"]}, path_length={item["a"]["path_length"]:.3f}')
        print(f'{LABEL_B}: success={b_ok}, answer={item["b"]["final_answer_text"]}, response_length={item["b"]["response_length"]}, path_length={item["b"]["path_length"]:.3f}')
        print(f'[{LABEL_A} response preview]')
        print(item['a']['response'][:900])
        print(f'[{LABEL_B} response preview]')
        print(item['b']['response'][:900])
        shown += 1
        if shown >= n:
            break

show_disagreements(matched, n=5)

try:
    import umap
    have_umap = True
except Exception as exc:
    have_umap = False
    print('UMAP is unavailable:', exc)

if have_umap:
    reducer_a = umap.UMAP(
        n_neighbors=min(15, max(2, len(matched) - 1)),
        min_dist=0.1,
        metric='euclidean',
        random_state=RANDOM_SEED,
    )
    reducer_b = umap.UMAP(
        n_neighbors=min(15, max(2, len(matched) - 1)),
        min_dist=0.1,
        metric='euclidean',
        random_state=RANDOM_SEED,
    )
    mean_umap_a = reducer_a.fit_transform(mean_a)
    mean_umap_b = reducer_b.fit_transform(mean_b)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, coords, labels, title in [
        (axes[0], mean_umap_a, labels_a, f'{LABEL_A}: mean-pooled trajectory UMAP'),
        (axes[1], mean_umap_b, labels_b, f'{LABEL_B}: mean-pooled trajectory UMAP'),
    ]:
        failure = labels == 0
        success = labels == 1
        if failure.any():
            ax.scatter(coords[failure, 0], coords[failure, 1], s=30, alpha=0.7, c='#d1495b', label='failure')
        if success.any():
            ax.scatter(coords[success, 0], coords[success, 1], s=30, alpha=0.7, c='#2a9d8f', label='success')
        ax.set_title(title)
        ax.set_xlabel('UMAP1')
        ax.set_ylabel('UMAP2')
        ax.grid(alpha=0.2)
        ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
try:
    import umap
    have_umap = True
except Exception as exc:
    have_umap = False
    print('UMAP is unavailable:', exc)

if have_umap:
    reducer = umap.UMAP(
        n_neighbors=min(15, max(2, len(matched) - 1)),
        min_dist=0.1,
        metric='euclidean',
        random_state=RANDOM_SEED,
    )
    mean_umap = reducer.fit_transform(combined_mean)

    plt.figure(figsize=(7, 6))
    plt.scatter(mean_umap[:n, 0], mean_umap[:n, 1], s=30, alpha=0.7, c='#1f77b4', label=LABEL_A)
    plt.scatter(mean_umap[n:, 0], mean_umap[n:, 1], s=30, alpha=0.7, c='#ff7f0e', label=LABEL_B)
    plt.title('Mean-pooled trajectory UMAP: model comparison')
    plt.xlabel('UMAP1')
    plt.ylabel('UMAP2')
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()